# SimSiam v2 訓練與資料預處理管線 (Jupyter Notebook 版)

此 Notebook 將 `v2/prepare_data.py` 與 `v2/train.py` 的流程整合，讓您能夠：
1. 以視覺化的方式確認資料前處理的結果（如連通元件分析、Logo 移除）。
2. 在同一環境下執行訓練與評估，並查看 Loss 曲線與評估指標。

**注意**: 請確保核心 (Kernel) 的執行目錄位於專案的根目錄 `Engineering_Image_Retrieval_System_Training` 或 `v2` 內。


In [1]:
import sys
import os
from pathlib import Path

# --- 確保專案根目錄與 v2 目錄在 Python Path 內 ---
# 假設目前的 Notebook 位於 v2 目錄內
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "v2" else notebook_dir

# 確保 v2 的 src 資料夾優先權最高，避免被根目錄的 src 遮蔽
v2_dir = project_root / "v2"
if str(v2_dir) not in sys.path:
    sys.path.insert(0, str(v2_dir))

# 切換 CWD 到專案根目錄，使所有相對路徑（data/, dataset/, outputs/）正常對齊
os.chdir(project_root)

print(f"Project Root: {project_root}")
print(f"Current Working Directory: {Path.cwd()}")

# --- 載入依賴套件 ---
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image

# 匯入 v2/src 的自訂模組
from src.config import AppConfig, DataConfig, ModelConfig, TrainingConfig, ExperimentConfig
from src.logger import setup_logging, get_logger
from src.training.timer import TimerCollection
from src.experiment.tracker import ExperimentTracker

# 繪圖中文設定
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK TC', 'Microsoft JhengHei', 'PingFang HK', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# 初始化 Logger
setup_logging(level="INFO", use_rich=True)
logger = get_logger("NotebookPipeline")


## 1. 參數設定 (Configuration)

我們不讀取 YAML，而是直接在此 Cell 中初始化 `AppConfig`，方便動態修改參數。


In [ ]:
# 定義設定參數
cfg = AppConfig(
    data=DataConfig(
        # 資料集來源 (這裡可以直接指定您預先放好的壓縮檔或目錄)
        # raw_zip_path="data/raw.zip", 
        skip_extraction=True,          # 範例：跳過解壓 (若已有 pdfs)
        skip_pdf_conversion=False,     # 是否執行 PDF 轉檔
        skip_preprocessing=False,      # 是否執行連通元件前處理
        n_runs=1,
        
        # 前處理參數
        use_connected_components=True,
        preprocess_top_n=5,
        preprocess_max_bbox_ratio=0.8,
        remove_gifu_logo=True,
        
        # 標註資料 (用於評估)
        labeled_data_path="data/converted_labeled_images"
    ),
    model=ModelConfig(
        backbone="resnet18",
        in_channels=1,
    ),
    training=TrainingConfig(
        epochs=10,                      # 測試用：設短一點
        batch_size=256,
        lr=3e-4,
        img_size=512,
        use_amp=True,
    ),
    experiment=ExperimentConfig(
        output_dir="outputs/notebook_exp",
        exp_name="simsiam_nb",
        eval_freq=5,
    )
)

# 驗證設定
cfg.validate()
logger.info("Configuration validated.")


## 2. 階段一：資料準備與預處理 (Data Preparation)

對應原來的 `prepare_data.py`。


In [3]:
from src.data.extraction import extract_archive
from src.data.pdf_converter import convert_pdfs_to_images
from src.data.preprocessing import PreprocessConfig, preprocess_images
from src.data.splitter import split_dataset

timers = TimerCollection()
d = cfg.data

# Step 1: 解壓縮
logger.info("=== Step 1: 解壓縮 ===")
if d.raw_zip_path and not d.skip_extraction:
    extract_archive(archive_path=d.raw_zip_path, output_dir=d.raw_pdf_dir, skip=False)
else:
    logger.info("跳過解壓縮 (skip_extraction=True 或是未提供 zip)")

# Step 2: PDF -> Image
logger.info("=== Step 2: PDF 轉影像 ===")
if not d.skip_pdf_conversion:
    # 確保目錄存在或有 PDF
    if Path(d.raw_pdf_dir).exists():
        convert_pdfs_to_images(
            pdf_dir=d.raw_pdf_dir,
            output_dir=d.converted_image_dir,
            dpi=d.pdf_dpi, max_workers=d.pdf_max_workers,
            skip=False, preserve_structure=False
        )
    else:
        logger.warning(f"找不到 PDF 來源目錄: {d.raw_pdf_dir}")


### 視覺化 1: 前處理預覽 (Preprocessing Preview)
沿用 `src/analysis/preview.py` 的邏輯，在 Notebook 中直接顯示影像的前處理過程。


In [4]:
from src.analysis.preview import PreprocessingPreview
import IPython.display as display_img

logger.info("=== 前處理預覽 ===")

preview_output = Path(cfg.experiment.output_dir) / "preview"
preview_output.mkdir(parents=True, exist_ok=True)

# 組合前處理參數字典
params = {
    "top_n": d.preprocess_top_n,
    "max_bbox_ratio": d.preprocess_max_bbox_ratio,
    "padding": d.preprocess_padding,
    "use_connected_components": d.use_connected_components,
    "use_topology_analysis": d.use_topology_analysis,
    "use_topology_pruning": d.use_topology_pruning,
    "topology_pruning_iters": d.topology_pruning_iters,
    "topology_pruning_ksize": d.topology_pruning_ksize,
    "min_simple_area": d.min_simple_area,
    "remove_gifu_logo": d.remove_gifu_logo,
    "logo_template_path": d.logo_template_path,
    "logo_mask_region": d.logo_mask_region,
    "min_bbox_area": d.preprocess_min_bbox_area,
    "img_size": cfg.training.img_size,
}

# 執行預覽 (隨機抽取 3 張)
# 確保我們有來源影像
if list(Path(d.converted_image_dir).glob("*.*")):
    preview = PreprocessingPreview(
        input_dir=d.converted_image_dir,
        n_samples=3,
        output_dir=str(preview_output),
        params=params,
        seed=42,
        figure_dpi=100
    )
    # 我們可以擷取產生的圖片並在 Notebook 中顯示
    preview.run()
    
    # 顯示產生的預覽圖
    for img_path in preview_output.glob("*.png"):
        print(f"Preview: {img_path.name}")
        display(display_img.Image(filename=str(img_path)))
else:
    logger.warning("尚無影像可供預覽。請先確保轉換步驟已執行。")


In [5]:
# Step 3: 正式執行影像前處理
logger.info("=== Step 3: 影像前處理 ===")

prep_cfg = PreprocessConfig(
    input_dir=d.converted_image_dir,
    output_root=d.preprocessed_image_dir,
    max_workers=d.preprocess_max_workers,
    top_n=d.preprocess_top_n,
    max_bbox_ratio=d.preprocess_max_bbox_ratio,
    min_bbox_area=d.preprocess_min_bbox_area,
    padding=d.preprocess_padding,
    use_connected_components=d.use_connected_components,
    use_topology_analysis=d.use_topology_analysis,
    use_topology_pruning=d.use_topology_pruning,
    topology_pruning_iters=d.topology_pruning_iters,
    topology_pruning_ksize=d.topology_pruning_ksize,
    min_simple_area=d.min_simple_area,
    remove_gifu_logo=d.remove_gifu_logo,
    logo_template_path=d.logo_template_path,
    logo_mask_region=d.logo_mask_region,
)

if not d.skip_preprocessing:
    preprocess_images(prep_cfg, skip=False)
else:
    logger.info("跳過正式前處理 (skip_preprocessing=True)")

# Step 4: 資料集分割
logger.info("=== Step 4: 資料集分割 ===")
for run_idx in range(d.n_runs):
    seed = d.base_seed + run_idx
    run_name = f"Run_{run_idx + 1:02d}_Seed_{seed}"
    n_train, n_test = split_dataset(
        source_root=d.preprocessed_image_dir,
        output_root=d.dataset_dir,
        run_name=run_name,
        split_ratio=d.split_ratio,
        seed=seed,
    )
    logger.info(f"  [{run_name}] train={n_train}, val/test={n_test}")


## 3. 階段二：模型訓練 (Model Training)

對應原來的 `train.py`。包含資料載入、訓練迴圈，以及動態繪圖 Callback。


In [6]:
from src.dataset.dataloader import create_dataloaders
from src.dataset.gpu_transforms import GPUAugmentation
from src.dataset.labeled_dataset import LabeledImageDataset
from src.model.simsiam import SimSiam
from src.training.trainer import Trainer
from src.training.checkpoint import CheckpointManager
from src.evaluation.evaluator import evaluate_model
import torch.backends.cudnn as cudnn

# 視覺化：使用 IPython display 更新 Loss 曲線
from IPython.display import clear_output

tracker = ExperimentTracker(config=cfg, timers=timers)
device = "cuda" if torch.cuda.is_available() else "cpu"
cudnn.benchmark = True if device == "cuda" else False

t = cfg.training
m = cfg.model
run_idx = 0
seed = d.base_seed

# 1. 載入 Labeled 資料集 (供檢索評估用)
labeled_ds = None
if Path(d.labeled_data_path).exists():
    labeled_ds = LabeledImageDataset(
        root=Path(d.labeled_data_path),
        use_preprocessing=cfg.training.use_preprocessing,
        use_invert=True,
        use_preprocessing=cfg.training.use_preprocessing,
        use_invert=True,
        img_size=t.img_size,
        in_channels=m.in_channels,
    )

# 2. 建立 DataLoader
run_dir_name = f"Run_{run_idx + 1:02d}_Seed_{seed}"
train_path = Path(d.dataset_dir) / run_dir_name / d.train_subpath
val_path = Path(d.dataset_dir) / run_dir_name / d.test_subpath

train_loader, val_loader, n_train, n_val = create_dataloaders(
    train_path, val_path, t, in_channels=m.in_channels, seed=seed,
    use_gpu_augmentation=(t.use_gpu_augmentation and device=="cuda")
)

# 3. 初始化模型、優化器
torch.manual_seed(seed)
model = SimSiam(
    backbone=m.backbone, proj_dim=m.proj_dim, proj_hidden=m.proj_hidden,
    pred_hidden=m.pred_hidden, pretrained=m.pretrained, in_channels=m.in_channels,
).to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=t.lr, momentum=0.9, weight_decay=t.weight_decay)

# Scheduler (Cosine Annealing)
steps_per_epoch = len(train_loader)
total_iters = steps_per_epoch * t.epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_iters)

scaler = torch.amp.GradScaler(device="cuda", enabled=(device == "cuda" and t.use_amp))

# 4. 初始化 GPU Augmentation
gpu_aug = GPUAugmentation(img_size=t.img_size, use_augmentation=t.use_augmentation, in_channels=m.in_channels).to(device) if (t.use_gpu_augmentation and device == "cuda") else None

run_tracker = tracker.create_run(run_dir_name)
ckpt_mgr = CheckpointManager(ckpt_dir=run_tracker.run_dir / "checkpoints", save_freq=cfg.experiment.save_freq)

trainer = Trainer(
    model=model, optimizer=optimizer, scheduler=scheduler, scaler=scaler,
    checkpoint_mgr=ckpt_mgr, device=device, grad_clip=t.grad_clip, gpu_aug=gpu_aug,
)

# 用來記錄繪圖資訊
history = {"epoch": [], "train_loss": [], "val_loss": [], "macro_mAP": []}

def live_plot_callback(metrics: dict, current_model: torch.nn.Module):
    epoch = metrics.get("epoch", 0)
    history["epoch"].append(epoch)
    history["train_loss"].append(metrics.get("train_loss", 0))
    history["val_loss"].append(metrics.get("val_loss", 0))
    
    # 評估 (KNN)
    should_eval = (epoch % cfg.experiment.eval_freq == 0) or (epoch == t.epochs)
    eval_score = None
    if labeled_ds is not None and should_eval:
        eval_metrics = evaluate_model(current_model, labeled_ds, device, cfg.data.eval_top_k_values, t.batch_size, t.num_workers)
        eval_score = eval_metrics.get("macro_mAP")
        metrics.update(eval_metrics)
        
    # 維持陣列長度一致，未評估則用 None 補齊
    if eval_score is not None:
        history["macro_mAP"].append(eval_score)
    else:
        history["macro_mAP"].append(None)
    
    run_tracker.log_epoch(metrics)
    
    # 動態繪圖
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(history["epoch"], history["train_loss"], label="Train Loss")
    ax1.plot(history["epoch"], history["val_loss"], label="Val Loss")
    ax1.set_title("Training & Validation Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)
    
    # 畫 Precision
    valid_epochs = [e for e, s in zip(history["epoch"], history["macro_mAP"]) if s is not None]
    valid_scores = [s for s in history["macro_mAP"] if s is not None]
    if valid_scores:
        ax2.plot(valid_epochs, valid_scores, marker='o', color='green', label="Macro-mAP (LOO)")
        ax2.set_title("Retrieval Evaluation (Macro-mAP)")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Macro-mAP")
        ax2.legend()
        ax2.grid(True)
        
    plt.tight_layout()
    plt.show()
    
    return eval_score if eval_score is not None else -float('inf')



In [ ]:
# 開始訓練
logger.info("=== 開始模型訓練 ===")
try:
    # 這邊因為 Notebook 環境，我們跑比較少 epoch 作為範例
    result = trainer.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=t.epochs,
        start_epoch=1,
        epoch_callback=live_plot_callback,
    )
    logger.info(f"訓練完成！Best Score: {result.get('best_score')}")
except Exception as e:
    logger.error(f"訓練發生錯誤: {e}")


## 4. 階段三：最終評估 (Evaluation)
匯出檢索結果統計報表。


In [ ]:
from src.experiment.reporter import generate_run_reports
import pandas as pd

# 產生訓練報告 CSV/Markdown
df = run_tracker.get_logs_dataframe()
generate_run_reports(df, run_tracker.run_dir, run_dir_name)

# 顯示最後幾筆訓練紀錄
display(df.tail())

logger.info(f"所有記錄已儲存於: {run_tracker.run_dir}")
